# 03 · Line profiles and optical depth

For every line j the gas at x absorbs the photon labelled by the observer's line-centred velocity v_† at the local
wavelength λ'_j = λ_j e^{v_†/c} a(x)/a_† D⁻¹(x) (notes `sec_flux.tex`, Eq. velocity_units_normalized), i.e. at the
frequency offset ν_j = c(1/λ' − 1/λ_j). The profile is a Voigt function φ = V(ν; σ, γ) with
σ = b/(√2 λ_j), b = √(2k_B T/m) + b_turb and γ = (A_21 + A_coll)/4π (Eq. line_profile), and

  τ_j(v_†) = C_j f_j Σ_i w_i n_com,i a_i⁻² φ_ij(v_†),  F = e^{−τ}  (Eq. tau_straight_los, discretised).

Runtime: ~40 s.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, ENTITY_COLORS as EC, MODE_STYLES as MS, TOL_BRIGHT as TOL, add_panel_letters
use_style()
%matplotlib inline

In [ ]:
from skewerjax.core import (LineBundle, LINES, voigt as V, line_wavelength_grid_A, frequency_offset_GHz, line_profiles,
                            compute_optical_depth, los_samples, resolution_report, wrap_to_ref_line, SkewerBundle, Fields,
                            ScalarField, VectorField, LogRamp, VelocityRamp, Constant, plane_points, analytic as an, phys)
lines = LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"], ref_line_id="HI-LyA")
print(f"{len(LINES)} transitions in lines_table.dat")
for l, dv in zip(lines.lines, np.asarray(lines.delta_v)):
    print(f"  {l.id:11s} lambda0 = {l.lambda0_A:9.4f} A  f = {l.f_osc:.4f}  A21 = {l.A21_Hz:.3e} Hz  m = {l.mass_amu:7.3f} amu  delta_v = {dv:9.1f} km/s")

## The Voigt function

Three implementations of the same V(x; σ, γ): `"ExoJAX"` (default; Humlíček-type, jit/grad), `"convolution"`
(G * L by FFT on the grid; needs a grid that resolves γ) and `"scipy_wofz"` (NumPy reference, not traceable).

In [ ]:
x = jnp.linspace(-30, 30, 2401)
fig, (ax, axr) = plt.subplots(2, 1, figsize=(13, 9), sharex=True, gridspec_kw={"height_ratios": [3, 1]}, layout="constrained")
for (s, g), c in zip([(1.0, 0.01), (1.0, 1.0), (0.3, 3.0)], [TOL["blue"], TOL["green"], TOL["red"]]):
    ref = V.voigt_scipy(np.asarray(x), np.asarray(s), np.asarray(g))
    ax.semilogy(x, ref, color=c, **MS["reference"])
    for m, sty in (("ExoJAX", MS["default"]), ("convolution", MS["matched"])):
        got = np.asarray(V.voigt_exojax(x, s, g) if m == "ExoJAX" else V.voigt_convolution(x, s, g, oversample=4))
        ax.semilogy(x, got, color=c, **sty, label=f"{m}, σ = {s}, γ = {g}" if s == 1.0 and g == 0.01 or m == "ExoJAX" else None)
        axr.plot(x, got / ref - 1, color=c, **sty)
ax.set(ylim=(1e-6, 2), ylabel="V(x; σ, γ)"); ax.legend(fontsize=13)
axr.set(xlabel="x", ylabel="/ wofz − 1", yscale="symlog", ylim=(-1, 1)); axr.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda y, p: f"{y:g}"))
add_panel_letters([ax, axr]); fig.suptitle("Voigt: ExoJAX (solid) and FFT convolution (dashed) on scipy's wofz (light)", fontsize=22); plt.show()

ExoJAX agrees with wofz to < 10⁻⁶ in the core; the convolution is exact (10⁻¹⁴) when the fine grid resolves γ and degrades at the grid ends and for γ below the grid spacing (`scripts/validation/check_voigt.py`).

## Line profiles at points

`line_profiles(events, directions, v, a_†, lines, fields)` evaluates φ_ij for every line and point: a hotter gas
broadens the lines, a peculiar velocity along the photon shifts them (to larger v for gas moving towards the
observer... in the frame of the photon direction).

In [ ]:
pts = plane_points(1e-3, "y", 5e-4, 1, 5)[0]
events = jnp.concatenate([jnp.ones((5, 1)), pts], -1)
v = jnp.linspace(-400, 400, 801)
fig, axs = plt.subplots(1, 2, figsize=(20, 6), layout="constrained")
for ax, (Tlo, Thi, vlo, vhi, title) in zip(axs, ((1e3, 1e6, 0.0, 0.0, r"$T$ from $10^3$ to $10^6$ K"), (1e4, 1e4, -300.0, 300.0, r"$v_x$ from −300 to +300 km/s, $T = 10^4$ K"))):
    sp = {"temperature": ScalarField(LogRamp(Tlo, Thi, 1e-3, 3)), "velocity": VectorField(VelocityRamp(vlo, vhi, 1e-3)),
          "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0)), "number_density": ScalarField(Constant(1.0))}
    ev = events.at[:, 3].set(jnp.linspace(0, 1e-3, 5)) if Tlo != Thi else events.at[:, 1].set(jnp.linspace(0, 1e-3, 5))
    phi = np.asarray(line_profiles(ev, jnp.broadcast_to(jnp.array([1.0, 0, 0]), (5, 3)), v, 1.0, LineBundle(["HI-LyA"]), Fields({"HI": sp})))[0]
    for i in range(5):
        ax.semilogy(v, phi[i], color=list(TOL.values())[i])
    ax.set(xlabel=r"$v_\dagger$ [km/s]", ylabel=r"$\phi$ [ns]", ylim=(1e-8, None), title=title)
add_panel_letters(axs); fig.suptitle(r"Ly$\alpha$ profiles: temperature sets the width, the line-of-sight velocity the centre", fontsize=22); plt.show()

The detuning uses the exact exponential mapping by default; `detuning="linear"` is fake_spectra's first-order version (matched mode, notebook 06). `velocity_period=P` folds the detuning into [−P/2, P/2) for periodic boxes.

## Optical depth: checks against closed forms

Static uniform gas (a = 1, no velocities) has closed forms (notes `app_analytical_solutions.tex`): a Lorentzian for
T = 0 and a Gaussian for A_21 = 0 (the `null…` lines of the table). Lines of one species share the fields.

In [ ]:
Lk = 1e-3
def fields(T, n):
    sp = lambda: {"temperature": ScalarField(LogRamp(*T, Lk, 3)), "velocity": VectorField(VelocityRamp(0.0, 0.0, Lk)),
                  "number_density": ScalarField(LogRamp(*n, Lk, 2)), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
    return Fields({"HI": sp(), "SiIII": sp()})
null = LineBundle(["nullHI-LyA", "nullHI-LyB", "nullSiIII1206"])
pts = plane_points(Lk, "y", Lk / 2, 4, 32)                                       # 4 skewers, temperature rising with z
sb = SkewerBundle(spatial_points=pts, manual_scale_factors=jnp.ones_like(pts[None, ..., 0]))
v = jnp.linspace(-3000.0, 3000.0, 1024)
F_th = fields((1e4, 1e8), (1e-7, 1e-7))
tau = compute_optical_depth(sb, v, null, F_th, a_dagger=1.0)
T_sk = F_th["HI"]["temperature"](sb.quadri_vectors[0])[:, 0]
wl = line_wavelength_grid_A(v, null, 1.0, 1.0)
ref = jnp.stack([an.tau_static_thermal(wl[j], null.lambda0_A[j:j+1], null.mass_amu[j:j+1], null.f_osc[j:j+1], sb.a0, Lk * jnp.ones(4),
                                       1e-7 * jnp.ones(4), T_sk)[0] for j in range(3)])
F, Fa = np.exp(-np.asarray(tau)), np.exp(-np.asarray(ref))
fig, axs = plt.subplots(2, 3, figsize=(22, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
for j, l in enumerate(lines.lines):
    for k in range(4):
        c = list(TOL.values())[k]
        axs[0, j].plot(v, Fa[j, k], color=c, **MS["reference"]); axs[0, j].plot(v, F[j, k], color=c, **MS["default"], label=f"T = {float(T_sk[k]):.0e} K" if j == 0 else None)
        axs[1, j].plot(v, F[j, k] - Fa[j, k], color=c)
    axs[0, j].set_title(l.name); axs[1, j].set_xlabel(r"$v_\dagger$ [km/s]")
axs[0, 0].set_ylabel("F"); axs[1, 0].set_ylabel(r"$F - F_{\rm analytic}$"); axs[0, 0].legend(fontsize=12)
add_panel_letters(axs); fig.suptitle("Static thermal lines: SkewerJAX (solid) on the analytic Gaussians (light); residuals < 10⁻⁸", fontsize=22); plt.show()

Radiation-dominated expansion and the convergence with the sampling are checked in `scripts/validation/check_optical_depth_analytic.py`.

## Options and the layers underneath

`compute_optical_depth(bundle, v, lines, fields, *, a_dagger, ii_cosmology, line_elements, voigt, detuning,
velocity_period, resolution_tolerance, v_chunk, batch_size)` is three pure layers:
`los_samples` → `LoSSamples` (every field at every point, a pytree; the boundary used by `skewerjax.simulations`),
`optical_depth_from_samples` (jit; `lax.map` over skewers, `batch_size` of them vmapped at once, `v_chunk` velocities
per call) and `optical_depth_kernel` (one skewer). `resolution_report` says whether the points resolve the lines
(notes `app_numerical_resolution.tex`).

In [ ]:
F_moving = Fields({sp: {"temperature": ScalarField(LogRamp(2e4, 2e4, Lk)), "velocity": VectorField(VelocityRamp(-300.0, 300.0, Lk)),
                         "number_density": ScalarField(LogRamp(1e-6, 1e-6, Lk)), "turbulence": ScalarField(Constant(0.0)),
                         "collisional": ScalarField(Constant(0.0))} for sp in ("HI", "SiIII")})
pts_x = jnp.stack([jnp.linspace(0, Lk, 1024), jnp.full(1024, Lk / 2), jnp.full(1024, Lk / 2)], -1)[None]   # along x: v_x ramps
for n_pt in (1024, 64):
    sbx = SkewerBundle(spatial_points=pts_x[:, ::1024 // n_pt], manual_scale_factors=jnp.ones((1, 1, n_pt)))
    rep = resolution_report(los_samples(sbx, F_moving, lines, a_dagger=1.0), lines, 0.25)
    print(f"{n_pt:4d} points: max |dv_c| / width per line = {np.round(rep['max_ratio'], 2)} -> {'resolved' if rep['ok'] else 'UNDER-RESOLVED (compute_optical_depth warns)'}")
sbx = SkewerBundle(spatial_points=pts_x, manual_scale_factors=jnp.ones((1, 1, 1024)))
v = jnp.linspace(-600, 2900, 1400)
lya_si = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
tau = compute_optical_depth(sbx, v, lya_si, F_moving, a_dagger=1.0)
tau_ref = wrap_to_ref_line(tau, v, lya_si)
fig, axs = plt.subplots(1, 2, figsize=(20, 6), layout="constrained", sharey=True)
for j, l in enumerate(lya_si.lines):
    axs[0].plot(v, np.exp(-tau[j, 0]), color=EC[l.id], label=l.id); axs[1].plot(v, np.exp(-tau_ref[j, 0]), color=EC[l.id])
axs[0].set(xlabel=r"$v_\dagger$ [km/s], each line in its own frame", ylabel="F", title="the same gas, two lines")
axs[1].set(xlabel=r"$v$ [km/s] on the Ly$\alpha$ frame (wrap_to_ref_line)", title=r"Si III moves by $\Delta v$ = 2270 km/s")
axs[0].legend(); add_panel_letters(axs); plt.show()

## JAX: jit, vmap, grad

Every function above is pure; the containers are pytrees. For example, the derivative of the mean flux with respect
to the density of the gas:

In [ ]:
def mean_flux(n0):
    f = Fields({sp: {"temperature": ScalarField(LogRamp(2e4, 2e4, Lk)), "velocity": VectorField(VelocityRamp(-300.0, 300.0, Lk)),
                     "number_density": ScalarField(Constant(n0)), "turbulence": ScalarField(Constant(0.0)),
                     "collisional": ScalarField(Constant(0.0))} for sp in ("HI", "SiIII")})
    return jnp.exp(-compute_optical_depth(sbx, v, lya_si, f, a_dagger=1.0, resolution_tolerance=None)[0]).mean()
n0 = 1e-6
print(f"<F> = {float(mean_flux(n0)):.4f}, d<F>/d ln n = {float(jax.grad(mean_flux)(n0) * n0):.4f}, "
      f"finite difference {float((mean_flux(n0 * 1.001) - mean_flux(n0 * 0.999)) / 0.002):.4f}")